# Day2 - Bronze: Exchange Rates

**Objetivo:** Criar a tabela Bronze `exchange_rates` a partir dos arquivos JSON salvos na camada Landing e carregar os dados com idempotência.

| Item | Valor |
|---|---|
| **Camada** | Bronze |
| **Entrada** | `/Volumes/{catalog}/landing/files/exchangerates/*.json` |
| **Saída** | `{catalog}.bronze.exchange_rates` |
| **Ordem no pipeline** | Após `Day1-Ingestao-Exchange-Rates` (Landing) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Carga Bronze — CREATE OR REPLACE TABLE AS SELECT (CTAS)

Lê todos os arquivos JSON da Landing, faz o explode do mapa de moedas e cria a tabela Bronze em uma única operação atômica (`CREATE OR REPLACE TABLE AS SELECT`). Os comentários das colunas são adicionados via `COMMENT ON COLUMN` logo após o CTAS. Reexecutar recria a tabela do zero com os dados mais recentes — sem `TRUNCATE` nem `INSERT` separados.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`bronze`.`exchange_rates`
USING DELTA
COMMENT 'Exchange rates of the world'
AS
    SELECT
        base,
        data_base,
        moeda,
        valor,
        from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS updated
    FROM (
        SELECT
            from_utc_timestamp(from_unixtime(timestamp), 'America/Sao_Paulo') AS data_extracao,
            base,
            TO_DATE(date) AS data_base,
            rates
        FROM read_files(
            '/Volumes/${catalog}/landing/files/exchangerates/*.json',
            format => 'json',
            multiLine => true)
    ) 
    LATERAL VIEW EXPLODE(FROM_JSON(TO_JSON(rates), 'MAP<STRING, DOUBLE>')) AS moeda, valor;

COMMENT ON COLUMN `${catalog}`.`bronze`.`exchange_rates`.base IS 'Moeda base (ex: EUR)';
COMMENT ON COLUMN `${catalog}`.`bronze`.`exchange_rates`.data_base IS 'Data da cotação';
COMMENT ON COLUMN `${catalog}`.`bronze`.`exchange_rates`.moeda IS 'Moeda de destino';
COMMENT ON COLUMN `${catalog}`.`bronze`.`exchange_rates`.valor IS 'Valor da taxa de câmbio';
COMMENT ON COLUMN `${catalog}`.`bronze`.`exchange_rates`.updated IS 'Timestamp da última atualização';